In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
import datetime as dt
from calitp_data_analysis.sql import get_engine
from shared_utils import gtfs_utils_v2
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
agencies_to_find = [
    "Bay Area 511 BART Schedule",
    "Big Blue Bus Swiftly Schedule",
    "Caltrain Schedule",
    "Culver City Schedule",
    "Foothill Schedule",
    "Fresno Schedule",
    "Gold Coast Schedule",
    "Golden Gate Park Shuttle Schedule",
    "Bay Area 511 Golden Gate Transit Schedule",
    "Long Beach Schedule",
    "Riverside Schedule",
    "SamTrans Schedule",
    "San Diego Schedule"
]

agency_sql = ', '.join(f"'{agency}'" for agency in agencies_to_find)

In [5]:
trip_columns = [
    "feed_key", "trip_instance_key", "trip_id", "route_id", "route_type", "name",
    "route_short_name", "route_long_name", "service_date", 
    "trip_first_departure_datetime_local_tz", "trip_last_arrival_datetime_local_tz",
    "time_of_day", "service_hours", "num_stop_times", "num_distinct_stops_served",
    "frequencies_defined_trip", "is_gtfs_flex_trip", "is_entirely_demand_responsive_trip"
]

In [6]:
with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-18')
          AND name IN ({agency_sql})
    """
    trips_sunday_data = pd.read_sql(query, connection)

In [7]:
with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-21')
          AND name IN ({agency_sql})
    """
    trips_weekday_data = pd.read_sql(query, connection)

In [8]:
with db_engine.connect() as connection:
    query = f"""
        SELECT {', '.join(trip_columns)}
        FROM `cal-itp-data-infra.mart_gtfs.fct_scheduled_trips`
        WHERE service_date = DATE('2025-05-17')
          AND name IN ({agency_sql})
    """
    trips_saturday_data = pd.read_sql(query, connection)

In [9]:
trips_saturday_data.head(5)

,feed_key,trip_instance_key,trip_id,route_id,route_type,name,route_short_name,route_long_name,service_date,trip_first_departure_datetime_local_tz,trip_last_arrival_datetime_local_tz,time_of_day,service_hours,num_stop_times,num_distinct_stops_served,frequencies_defined_trip,is_gtfs_flex_trip,is_entirely_demand_responsive_trip
0,f6774d861953d4f4cdcffec95e2652c7,8962198727f2e3278f935c244148773f,1198100,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-17,2025-05-17 12:45:00,2025-05-17 13:27:00,midday,0.700000,42,42,False,False,False
1,f6774d861953d4f4cdcffec95e2652c7,2bf3c2a748649550a2e8bc28b65f1aa1,499100,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-17,2025-05-17 13:15:00,2025-05-17 13:58:00,midday,0.716667,42,42,False,False,False
2,f6774d861953d4f4cdcffec95e2652c7,4b99db9e7de6942abcd82f96e66ee730,1073100,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-17,2025-05-17 13:45:00,2025-05-17 14:28:00,midday,0.716667,42,42,False,False,False
3,f6774d861953d4f4cdcffec95e2652c7,b8674a68a5fd2ada304fe44fac3587b6,1142100,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-17,2025-05-17 09:07:00,2025-05-17 09:44:00,am_peak,0.616667,42,42,False,False,False
4,f6774d861953d4f4cdcffec95e2652c7,674df1ef6d4385af4e81f5f49da3edad,958100,1,3,Culver City Schedule,1,Washington Boulevard,2025-05-17,2025-05-17 15:45:00,2025-05-17 16:29:00,pm_peak,0.733333,42,42,False,False,False


In [10]:
# Get unique feed_keys from your trips_weekday_data
weekday_feed_key = trips_weekday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_weekday = ",".join(f"'{fk}'" for fk in weekday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-21')
          AND feed_key IN ({feed_keys_str_weekday})
    """
    stops_unique_weekday = pd.read_sql(query, connection)

In [11]:
# Get unique feed_keys from your trips_weekday_data
saturday_feed_key = trips_saturday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_saturday = ",".join(f"'{fk}'" for fk in saturday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-17')
          AND feed_key IN ({feed_keys_str_saturday})
    """
    stops_unique_saturday = pd.read_sql(query, connection)

In [12]:
# Get unique feed_keys from your trips_weekday_data
sunday_feed_key = trips_sunday_data['feed_key'].unique().tolist()

# Convert to SQL-friendly string for IN clause
feed_keys_str_sunday = ",".join(f"'{fk}'" for fk in sunday_feed_key)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-17')
          AND feed_key IN ({feed_keys_str_sunday})
    """
    stops_unique_sunday = pd.read_sql(query, connection)

In [13]:
stops_unique_weekday.head(5)

,feed_key,stop_id,_feed_valid_from,n_hours_in_service,arrivals_early_am,arrivals_am_peak,arrivals_midday,arrivals_pm_peak,arrivals_evening,route_id_array,route_type_array,stop_key,tts_stop_name,pt_geom,stop_name,location_type,stop_desc
0,4e549244dd7ce383a3f4337f00134b27,908301,2025-05-09 03:06:16.096948+00:00,1,0,0,13,0,0,[Yellow-N],[1],242e01183fbe06394c8a57fd6a57e63a,None,POINT(-121.780373 37.995343),Antioch,0.0,None
1,4e549244dd7ce383a3f4337f00134b27,907103,2025-05-09 03:06:16.096948+00:00,1,0,0,0,0,9,[Yellow-S],[1],282477848f546fce0dde0358805a408e,None,POINT(-122.391895 37.616023),San Francisco International Airport,0.0,None
2,661ef844bdaa253e8b950740f76061b1,6130,2025-05-15 03:07:57.676759+00:00,4,0,0,0,28,0,[490],[3],7fda163f8bd16010613a619b31393aa6,None,POINT(-117.872481 34.107083),Grand Ave and Carter Dr E,0.0,None
3,661ef844bdaa253e8b950740f76061b1,831,2025-05-15 03:07:57.676759+00:00,4,0,0,0,28,0,[490],[3],254bfd357ca3435b61bcd261a664db53,None,POINT(-117.873101 34.121512),Base Line Rd and Grand Ave E,0.0,None
4,661ef844bdaa253e8b950740f76061b1,2633,2025-05-15 03:07:57.676759+00:00,5,0,0,0,35,0,[499],[3],cd4380a2c7f1c679b36e178a523ac7b6,None,POINT(-117.837496 34.069694),Via Verde Park and Ride E,0.0,None
